In [1]:
import polars as pl

# Load lazily to reduce RAM usage
df = pl.read_parquet("../data/malaysia_transactions.parquet")

In [2]:
columns_to_keep = [
    "date_time",
    "ofi_entity_id",
    "rfi_entity_id",
    "trxn_amount",
    "trxn_type",
    "trxn_channel"
]

df_clean = df.select(columns_to_keep)

df_clean.null_count()

date_time,ofi_entity_id,rfi_entity_id,trxn_amount,trxn_type,trxn_channel
u32,u32,u32,u32,u32,u32
0,438281,464000,0,665434,0


In [3]:
# Drop any rows with missing sender/receiver IDs
df_filtered = df_clean.filter(
    pl.col("ofi_entity_id").is_not_null() & 
    pl.col("rfi_entity_id").is_not_null()
)

# Fill missing `trxn_type` with fallback label
df_filtered = df_filtered.with_columns(
    pl.col("trxn_type").fill_null("Unknown")
)

In [4]:
df_filtered.null_count()

date_time,ofi_entity_id,rfi_entity_id,trxn_amount,trxn_type,trxn_channel
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


In [5]:
df_filtered.shape

(11396551, 6)

In [6]:
# Sort the dataframe by time
df_sorted = df_filtered.sort("date_time")

# check earliest and latest timestamp
print("Start:", df_sorted["date_time"][0])
print("End:", df_sorted["date_time"][-1])

Start: 2025-06-01 00:00:00
End: 2025-06-30 23:59:59


In [7]:
from collections import defaultdict

# Directed graph: entity_id → list of (target_entity, timestamp, amount, type)
graph = defaultdict(list)

# Risk score for each entity_id (float from 0 to 1)
entity_risk = defaultdict(float)

In [11]:
from datetime import datetime, timedelta

# Rule threshold
MAX_LAYERING_DEPTH = 4
MAX_TIME_GAP = timedelta(minutes=10)  # fast flow between nodes

# Utility: parse timestamp if it's not already datetime
def ensure_datetime(ts):
    return ts if isinstance(ts, datetime) else datetime.strptime(str(ts), "%Y-%m-%d %H:%M:%S")

# Recursive DFS to measure path depth
def layering_depth(entity, visited, current_time, depth=0):
    if depth >= MAX_LAYERING_DEPTH:
        return depth

    max_depth = depth
    for (target, t, amt, ttype) in graph[entity]:
        t = ensure_datetime(t)
        if target not in visited and (current_time - t) <= MAX_TIME_GAP:
            visited.add(target)
            new_depth = layering_depth(target, visited, t, depth + 1)
            max_depth = max(max_depth, new_depth)
            visited.remove(target)

    return max_depth

# Main processor per transaction
def process_transaction(sender, receiver, timestamp, amount, ttype):
    timestamp = ensure_datetime(timestamp)

    # ✅ First update graph BEFORE checking depth
    graph[sender].append((receiver, timestamp, amount, ttype))

    # Now check for layering depth
    visited = set([sender])
    depth = layering_depth(sender, visited, timestamp)

    # Apply risk if chain is deep enough
    if depth >= 2:
        risk_increase = min(1.0, 0.1 * depth)
        entity_risk[receiver] += risk_increase

In [9]:
row = df_sorted.row(0)
process_transaction(
    sender=row[1],          # ofi_entity_id
    receiver=row[2],        # rfi_entity_id
    timestamp=row[0],       # date_time
    amount=row[3],          # trxn_amount
    ttype=row[4]            # trxn_type
)

In [10]:
N = 100000  # start small, increase later if fast

for i in range(N):
    row = df_sorted.row(i)
    process_transaction(
        sender=row[1],
        receiver=row[2],
        timestamp=row[0],
        amount=row[3],
        ttype=row[4]
    )

# Inspect: how many entities have non-zero risk
non_zero_risk = {k: v for k, v in entity_risk.items() if v > 0}
print("Entities with non-zero risk:", len(non_zero_risk))

# Show top 10 riskiest entities
top_risky = sorted(non_zero_risk.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top risky entities:", top_risky)

Entities with non-zero risk: 0
Top risky entities: []


In [13]:
from datetime import datetime, timedelta

graph = defaultdict(list)
entity_risk = defaultdict(float)

base_time = datetime(2025, 6, 1, 12, 0, 0)

synthetic_chain = [
    ("E001", "E002", base_time, 1000.0, "Online Transfer"),
    ("E002", "E003", base_time + timedelta(minutes=1), 980.0, "Online Transfer"),
    ("E003", "E004", base_time + timedelta(minutes=2), 970.0, "Online Transfer"),
    ("E004", "E005", base_time + timedelta(minutes=3), 950.0, "Online Transfer"),
]

for s, r, t, a, tt in synthetic_chain:
    process_transaction(s, r, t, a, tt)

print("Risk on E002:", entity_risk["E002"])
print("Risk on E003:", entity_risk["E003"])
print("Risk on E004:", entity_risk["E004"])
print("Risk on E005:", entity_risk["E005"])


Risk on E002: 0.0
Risk on E003: 0.0
Risk on E004: 0.0
Risk on E005: 0.0
